<a href="https://colab.research.google.com/github/khyun8072/CAU_AISW_NLP/blob/main/Week4/251105_%E1%84%80%E1%85%AE%E1%86%ABAI_%E1%84%89%E1%85%B5%E1%86%AF%E1%84%89%E1%85%B3%E1%86%B8%E1%84%8C%E1%85%A1%E1%84%85%E1%85%AD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pre-trained 모델 구조별 실습: Encoder-only / Encoder–Decoder / Decoder-only

본 노트북은 **세 가지 Transformer 아키텍처**를 소규모 예제로 비교·실습한다.
- **Encoder-only (BERT)**: GLUE **SST-2** 감성 분석 + 파인튜닝 전후 비교 + **Self-Attention 맵 시각화**
- **Encoder–Decoder (T5)**: **CNN/DailyMail** 요약 + 파인튜닝 전후 비교 + **Cross-Attention 맵 시각화**
- **Decoder-only (GPT-2)**: **CNN/DailyMail** 요약 + 파인튜닝 전후 비교 + **Self-Attention 맵 시각화**

## 0. 환경 준비 (필수)
**목표:** 필요한 라이브러리를 설치/불러오고, 실행 환경(CPU/GPU)을 점검한다.

**설치 목록:** `transformers`, `datasets`, `accelerate`, `evaluate`, `sentencepiece`, `torch` (미설치 시)

> 주의: 대규모 모델/데이터는 GPU 환경에서 실행을 권장한다. CPU에서도 작동하나 시간이 소요될 수 있다.

In [ ]:
%%bash
python - <<'PY'
import sys
import importlib
pkgs = [
  ('transformers', None), ('datasets', None), ('accelerate', None),
  ('evaluate', None), ('sentencepiece', None)
]
missing = []
for name,_ in pkgs:
    try:
        importlib.import_module(name)
    except Exception:
        missing.append(name)
if missing:
    print('다음 패키지 설치 필요:', missing)
else:
    print('필요 패키지가 이미 설치되어 있습니다.')
PY

In [ ]:
# (필요 시) pip 설치 셀 — 주석 해제 후 사용
# %pip install -U transformers datasets accelerate evaluate sentencepiece
# # PyTorch는 환경에 맞게 설치 (https://pytorch.org/get-started/locally/) 참고

In [ ]:
# 러닝 환경 점검 (GPU 유무)
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

### 공통 유틸리티: 토큰 디코딩 및 Attention 시각화 도우미
**목표:** 모델별 Attention 텐서를 간단히 시각화하는 도우미 함수를 정의한다.

In [ ]:
from typing import List, Optional, Tuple
import numpy as np
import matplotlib.pyplot as plt

def plot_attention_heatmap(attn: np.ndarray,
                           x_tokens: List[str], y_tokens: Optional[List[str]] = None,
                           title: str = 'Attention Heatmap',
                           figsize: Tuple[int,int] = None):
    """
    attn: (seq_len_y, seq_len_x) 또는 (seq_len, seq_len)
    x_tokens: x축 토큰 리스트
    y_tokens: y축 토큰 리스트 (None이면 x_tokens 사용: self-attention)
    figsize: 직접 지정 가능 (None이면 자동 계산)
    """
    if y_tokens is None:
        y_tokens = x_tokens

    # 패딩 토큰([PAD]) 제거 - 실제 토큰만 시각화
    non_pad_x_indices = [i for i, tok in enumerate(x_tokens) if tok not in ['[PAD]', '', '<pad>']]
    non_pad_y_indices = [i for i, tok in enumerate(y_tokens) if tok not in ['[PAD]', '', '<pad>']]

    # 패딩 제거한 attention 행렬과 토큰 리스트
    attn_filtered = attn[np.ix_(non_pad_y_indices, non_pad_x_indices)]
    x_tokens_filtered = [x_tokens[i] for i in non_pad_x_indices]
    y_tokens_filtered = [y_tokens[i] for i in non_pad_y_indices]

    # 토큰 길이에 맞게 figsize 동적 조정 (더 합리적인 크기)
    if figsize is None:
        # 최대 크기 제한 추가
        fig_width = min(20, max(10, len(x_tokens_filtered) * 0.3))
        fig_height = min(12, max(6, len(y_tokens_filtered) * 0.3))
        figsize = (fig_width, fig_height)

    plt.figure(figsize=figsize)
    plt.imshow(attn_filtered, aspect='auto', cmap='viridis')
    plt.colorbar()

    # 토큰이 많으면 폰트 크기 줄이기
    fontsize = 9 if len(x_tokens_filtered) > 30 else 10

    # x축 토큰 (90도 회전)
    plt.xticks(range(len(x_tokens_filtered)), x_tokens_filtered, rotation=90, fontsize=fontsize)
    # y축 토큰
    plt.yticks(range(len(y_tokens_filtered)), y_tokens_filtered, fontsize=fontsize)

    plt.title(title, fontsize=12)
    plt.xlabel('Source Tokens', fontsize=10)
    plt.ylabel('Target Tokens', fontsize=10)
    plt.tight_layout()
    plt.show()

## 1. Encoder-only 모델: BERT로 SST-2 감성 분석
**목표:**
1) SST-2 데이터셋 샘플 확인 및 전처리
2) BERT의 **encoder-only** 구조 확인
3) 파인튜닝 전후 성능 비교
4) **Self-Attention 맵** 시각화

In [ ]:
# 1-1) 데이터 준비: GLUE/SST-2
from datasets import load_dataset
sst2 = load_dataset('glue', 'sst2')
sst2

In [ ]:
# 1-2) SST-2 데이터셋 샘플 확인
print("=== SST-2 Train 샘플 예시 (처음 5개) ===")
for i in range(5):
    sample = sst2['train'][i]
    print(f"\n[Sample {i+1}]")
    print(f"  Sentence: {sample['sentence']}")
    print(f"  Label: {sample['label']} ({'positive' if sample['label']==1 else 'negative'})")

print("\n\n=== SST-2 Validation 샘플 예시 (처음 3개) ===")
for i in range(3):
    sample = sst2['validation'][i]
    print(f"\n[Sample {i+1}]")
    print(f"  Sentence: {sample['sentence']}")
    print(f"  Label: {sample['label']} ({'positive' if sample['label']==1 else 'negative'})")

In [ ]:
# 1-3) 토크나이저 및 전처리 함수 정의
from transformers import AutoTokenizer
bert_checkpoint = 'google-bert/bert-base-uncased'
tokenizer_bert = AutoTokenizer.from_pretrained(bert_checkpoint)

def preprocess_bert(batch):
    # SST-2는 단일 문장 분류이므로 pair 없음
    return tokenizer_bert(batch['sentence'], truncation=True, padding='max_length', max_length=128)

sst2_enc = sst2.map(preprocess_bert, batched=True)
sst2_enc = sst2_enc.remove_columns(['sentence', 'idx'])
sst2_enc.set_format(type='torch')
sst2_enc

In [ ]:
# 1-4) 입력 구조 샘플 확인
sample = sst2_enc['validation'][0]
for k,v in sample.items():
    print(k, v.shape)
sample

In [ ]:
# 1-5) 파인튜닝 전 베이스 모델 로드 및 구조 확인 (encoder-only)
from transformers import BertForSequenceClassification
bert_base = BertForSequenceClassification.from_pretrained(bert_checkpoint, num_labels=2).to(device)
print('is_encoder_decoder:', bert_base.config.is_encoder_decoder)
print('architectures:', bert_base)
print('model_type:', bert_base.config.model_type)

In [ ]:
# 1-6) 파인튜닝 전 모델 평가 (Validation 셋의 일부로 테스트)
import torch
from torch.utils.data import DataLoader

# 평가 함수 정의
def evaluate_model(model, dataset, num_samples=100):
    model.eval()
    dataloader = DataLoader(dataset.select(range(min(num_samples, len(dataset)))), batch_size=8)

    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            inputs = {k: v.to(device) for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'token_type_ids']}
            labels = batch['label'].to(device)

            outputs = model(**inputs)
            predictions = torch.argmax(outputs.logits, dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = correct / total
    return accuracy

print("=== 파인튜닝 전 모델 평가 ===")
pre_finetune_acc = evaluate_model(bert_base, sst2_enc['validation'], num_samples=len(sst2_enc['validation']))
print(f"Validation Accuracy (전체): {pre_finetune_acc:.4f} ({pre_finetune_acc*100:.2f}%)")

In [ ]:
# 1-7) BERT 파인튜닝 (SST-2 데이터셋)
from transformers import TrainingArguments, Trainer
import evaluate

# 메트릭 로드
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 학습 설정 (소규모 학습)
training_args = TrainingArguments(
    output_dir='./results_bert_sst2',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
)

# 학습 데이터 일부만 사용 (빠른 실습을 위해)
train_dataset = sst2_enc['train']  # 5000개만 사용
eval_dataset = sst2_enc['validation']

# Trainer 초기화
trainer = Trainer(
    model=bert_base,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

print("=== BERT 파인튜닝 시작 ===")
trainer.train()

In [ ]:
# 1-8) 파인튜닝 후 모델 평가
from transformers import AutoModelForSequenceClassification

print("\n=== 학습된 모델 가중치 로드 ===")
# 학습 중 저장된 best model 로드
best_model_path = './results_bert_sst2'
bert_finetuned = AutoModelForSequenceClassification.from_pretrained(best_model_path)
bert_finetuned.to(device)
print(f"모델 로드 완료: {best_model_path}")

print("\n=== 파인튜닝 후 모델 평가 ===")
post_finetune_acc = evaluate_model(bert_finetuned, sst2_enc['validation'], num_samples=len(sst2_enc['validation']))
print(f"Validation Accuracy (전체): {post_finetune_acc:.4f} ({post_finetune_acc*100:.2f}%)")

print("\n=== 성능 비교 ===")
print(f"파인튜닝 전: {pre_finetune_acc*100:.2f}%")
print(f"파인튜닝 후: {post_finetune_acc*100:.2f}%")
print(f"향상도: {(post_finetune_acc - pre_finetune_acc)*100:.2f}%p")

In [ ]:
# 1-9) 파인튜닝 모델로 단일 배치 추론 및 Attention 추출
import torch
bert_base.eval()
sample_idx = 0
sample = sst2_enc['validation'][sample_idx]
original_sentence = sst2['validation'][sample_idx]['sentence']  # 원본 문장 가져오기

batch = {k: v.unsqueeze(0).to(device) for k,v in sample.items() if k in ['input_ids','attention_mask','token_type_ids']}
with torch.no_grad():
    outputs = bert_base(**batch, output_attentions=True, return_dict=True)
logits = outputs.logits
pred = torch.argmax(logits, dim=-1).item()
true_label = sample['label'].item()

print('\n=== 단일 샘플 추론 결과 ===')
print(f'Input sentence: "{original_sentence}"')
print('logits:', logits.cpu().numpy())
print('pred label:', pred, '(0=negative, 1=positive)')
print('true label:', true_label, '(0=negative, 1=positive)')
print('Correct!' if pred == true_label else 'Wrong!')

In [ ]:
# 1-10) Self-Attention 맵 시각화 (마지막 레이어, 전체 헤드 평균)
attentions = outputs.attentions  # tuple: (num_layers, batch, num_heads, seq_len, seq_len)
last_layer_attn = attentions[-1][0].detach().cpu().numpy()  # (num_heads, seq_len, seq_len)

# 모든 헤드의 평균
attn_mean = last_layer_attn.mean(axis=0)  # (seq_len, seq_len)

tokens = tokenizer_bert.convert_ids_to_tokens(batch['input_ids'][0].cpu().tolist())

print(f"\n=== Attention 시각화 정보 ===")
print(f"레이어 수: {len(attentions)}")
print(f"헤드 수: {last_layer_attn.shape[0]}")
print(f"시각화: 마지막 레이어의 {last_layer_attn.shape[0]}개 헤드 평균")

plot_attention_heatmap(attn_mean, tokens, tokens,
                       title=f'BERT Self-Attention (last layer, average of {last_layer_attn.shape[0]} heads)')

## 2. Encoder–Decoder 모델: T5로 CNN/DailyMail 요약
**목표:**
1) T5의 인코더–디코더 구조 확인
2) CNN/DailyMail 데이터셋 샘플 확인
3) 파인튜닝 전후 요약 품질 비교
4) **Cross-Attention 맵** 시각화

In [ ]:
# 2-1) 데이터 준비: CNN/DailyMail (v3.0.0)
from datasets import load_dataset
cnn = load_dataset('cnn_dailymail', '3.0.0')
print(cnn)

In [ ]:
# 2-2) CNN/DailyMail 데이터셋 샘플 확인
print("=== CNN/DailyMail Test 샘플 예시 #1 ===\n")
sample_1 = cnn['test'][0]

print("[Article (처음 400자)]")
print(sample_1['article'][:400])
print("\n...(생략)...\n")

print("[Highlights (정답 요약)]")
print(sample_1['highlights'])

print("\n\n=== CNN/DailyMail Test 샘플 예시 #2 ===\n")
sample_2 = cnn['test'][1]
print("[Article (처음 400자)]")
print(sample_2['article'][:400])
print("\n...(생략)...\n")

print("[Highlights (정답 요약)]")
print(sample_2['highlights'])

In [ ]:
# 2-3) T5 토크나이저/모델 로드 및 구조 확인
from transformers import T5ForConditionalGeneration, T5TokenizerFast
t5_ckpt = 't5-small'  # 가벼운 모델
tokenizer_t5 = T5TokenizerFast.from_pretrained(t5_ckpt)
t5_base = T5ForConditionalGeneration.from_pretrained(t5_ckpt).to(device)
print('is_encoder_decoder:', t5_base)
print('model_type:', t5_base.config.model_type)

In [ ]:
# 2-4) 파인튜닝 전 T5 요약 생성 테스트
import torch
prefix = 'summarize: '
test_article = cnn['test'][0]['article']
inputs = tokenizer_t5(prefix + test_article, return_tensors='pt', truncation=True, max_length=512).to(device)

print("=== 파인튜닝 전 T5 요약 결과 ===\n")
print("[원본 Article (처음 300자)]")
print(test_article[:300])
print("\n...(생략)...\n")

summary_ids_before = t5_base.generate(
    **inputs,
    max_length=120,
    num_beams=4,
    early_stopping=True
)
summary_before = tokenizer_t5.decode(summary_ids_before[0], skip_special_tokens=True)

print("[정답 Highlights]")
print(cnn['test'][0]['highlights'])

print("\n[파인튜닝 전 T5 생성 요약]")
print(summary_before)

In [ ]:
# 2-5) T5 파인튜닝 데이터 전처리
def preprocess_t5_summarization(examples):
    inputs = ["summarize: " + doc for doc in examples['article']]
    model_inputs = tokenizer_t5(inputs, max_length=512, truncation=True, padding='max_length')

    # 타겟 (요약) - 중요: padding token을 -100으로 설정해야 loss 계산에서 무시됨
    labels = tokenizer_t5(examples['highlights'], max_length=128, truncation=True, padding='max_length')

    # padding 토큰을 -100으로 변경 (PyTorch loss에서 무시됨)
    labels['input_ids'] = [
        [(token if token != tokenizer_t5.pad_token_id else -100) for token in label]
        for label in labels['input_ids']
    ]

    model_inputs['labels'] = labels['input_ids']

    return model_inputs

# 학습용 데이터 준비 (시간 절약을 위해 소량만 사용)
train_dataset_t5 = cnn['train'].select(range(1000)).map(preprocess_t5_summarization, batched=True)
eval_dataset_t5 = cnn['validation'].select(range(100)).map(preprocess_t5_summarization, batched=True)

train_dataset_t5.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_dataset_t5.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

print("T5 학습 데이터 준비 완료")
print(f"Train samples: {len(train_dataset_t5)}")
print(f"Eval samples: {len(eval_dataset_t5)}")

# 샘플 확인
sample = train_dataset_t5[0]
print(f"\n샘플 labels 확인 (처음 20개): {sample['labels'][:20]}")
print(f"패딩(-100) 개수: {(sample['labels'] == -100).sum()}")

In [ ]:
# 2-6) T5 파인튜닝 실행
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args_t5 = Seq2SeqTrainingArguments(
    output_dir='./results_t5_cnn',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs_t5',
    logging_steps=50,
    predict_with_generate=True,
    load_best_model_at_end=True,
)

trainer_t5 = Seq2SeqTrainer(
    model=t5_base,
    args=training_args_t5,
    train_dataset=train_dataset_t5,
    eval_dataset=eval_dataset_t5,
)

print("=== T5 파인튜닝 시작 ===")
trainer_t5.train()

In [ ]:
# 2-7) 파인튜닝 후 T5 요약 생성 및 비교
from transformers import AutoModelForSeq2SeqLM

print("\n=== 학습된 T5 모델 가중치 로드 ===")
# 학습 중 저장된 best model 로드
best_t5_model_path = './results_t5_cnn/checkpoint-375'  # 실제 체크포인트 경로로 변경 필요
t5_finetuned = AutoModelForSeq2SeqLM.from_pretrained(best_t5_model_path)
t5_finetuned.to(device)
print(f"T5 모델 로드 완료: {best_t5_model_path}")

print("\n=== 파인튜닝 후 T5 요약 결과 ===\n")
print("[원본 Article (처음 300자)]")
print(test_article[:300])
print("\n...(생략)...\n")

summary_ids_after = t5_finetuned.generate(
    **inputs,
    max_length=120,
    min_length=10,
    num_beams=4,
    length_penalty=1.0,
    early_stopping=True,
    return_dict_in_generate=True,
    output_attentions=True
)
summary_after = tokenizer_t5.decode(summary_ids_after.sequences[0], skip_special_tokens=True)

print("[정답 Highlights]")
print(cnn['test'][0]['highlights'])

print("\n[파인튜닝 전 T5 생성 요약]")
print(summary_before)

print("\n[파인튜닝 후 T5 생성 요약]")
print(summary_after)

In [ ]:
# 2-8) Cross-Attention 시각화 (파인튜닝 후 모델, 전체 헤드 평균)
# 전체 생성된 시퀀스의 cross-attention 시각화
gen_cross = summary_ids_after.cross_attentions  # 길이=생성된 토큰 수

# 마지막 레이어의 모든 생성 스텝의 cross-attention 수집
num_generated = len(gen_cross)
last_layer_idx = -1

# 각 생성 스텝에서 마지막 레이어의 헤드 평균을 추출
cross_attn_all_steps = []
for step_idx in range(num_generated):
    step_cross = gen_cross[step_idx][last_layer_idx][0]  # (num_heads, 1, src_len)
    step_cross_mean = step_cross.mean(dim=0)[0].detach().cpu().numpy()  # (src_len,)
    cross_attn_all_steps.append(step_cross_mean)

# (num_generated, src_len) 형태로 스택
cross_attn_matrix = np.stack(cross_attn_all_steps, axis=0)

src_tokens = tokenizer_t5.convert_ids_to_tokens(inputs['input_ids'][0].cpu().tolist())
tgt_tokens = tokenizer_t5.convert_ids_to_tokens(summary_ids_after.sequences[0].cpu().tolist())

print(f"\n=== Cross-Attention 시각화 정보 ===")
print(f"생성된 토큰 수: {num_generated}")
print(f"소스 시퀀스 길이: {len(src_tokens)}")
print(f"타겟 시퀀스 길이: {len(tgt_tokens)}")
print(f"시각화: 각 디코더 토큰이 인코더 토큰에 주는 attention")

plot_attention_heatmap(cross_attn_matrix, src_tokens, tgt_tokens,
                       title=f'T5 Cross-Attention (last layer, average of all heads)')

## 3. Decoder-only 모델: GPT-2로 CNN/DailyMail 요약 프롬프트 생성
**목표:**
1) GPT-2의 **decoder-only** 구조 확인
2) 파인튜닝 전후 요약 생성 비교
3) **Self-Attention 맵** 시각화

In [ ]:
# 3-1) GPT-2 모델/토크나이저 로드 및 구조 확인
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
gpt2_ckpt = 'gpt2'
tokenizer_gpt2 = GPT2TokenizerFast.from_pretrained(gpt2_ckpt)
tokenizer_gpt2.pad_token = tokenizer_gpt2.eos_token  # 패딩 토큰 지정
gpt2_base = GPT2LMHeadModel.from_pretrained(gpt2_ckpt).to(device)
print('is_encoder_decoder:', gpt2_base.config.is_encoder_decoder)
print('model_type:', gpt2_base)
print('n_layer:', gpt2_base.config.n_layer, 'n_head:', gpt2_base.config.n_head)

In [ ]:
# 3-2) 파인튜닝 전 GPT-2 요약 생성 테스트
prompt = (
    'Summarize the news article in 2-3 sentences.\n\nArticle: ' + test_article + '\n\nTL;DR:'
)
inputs_gpt2 = tokenizer_gpt2(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)

print("=== 파인튜닝 전 GPT-2 요약 결과 ===\n")
print("[원본 Article (처음 300자)]")
print(test_article[:300])
print("\n...(생략)...\n")

gen_out_before = gpt2_base.generate(
    **inputs_gpt2,
    max_length=inputs_gpt2['input_ids'].shape[1] + 100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer_gpt2.eos_token_id
)
generated_before = tokenizer_gpt2.decode(gen_out_before[0][inputs_gpt2['input_ids'].shape[1]:], skip_special_tokens=True)

print("[정답 Highlights]")
print(cnn['test'][0]['highlights'])

print("\n[파인튜닝 전 GPT-2 생성 요약]")
print(generated_before)

In [ ]:
# 3-3) GPT-2 파인튜닝 데이터 전처리
def preprocess_gpt2_summarization(examples):
    prompts = []
    for article, highlights in zip(examples['article'], examples['highlights']):
        prompt = f"Summarize the news article in 2-3 sentences.\n\nArticle: {article}\n\nTL;DR: {highlights}{tokenizer_gpt2.eos_token}"
        prompts.append(prompt)

    # GPT-2는 causal LM이므로 input과 label이 동일
    tokenized = tokenizer_gpt2(prompts, truncation=True, max_length=512, padding='max_length')
    tokenized['labels'] = tokenized['input_ids'].copy()

    return tokenized

# 학습용 데이터 준비 (시간 절약을 위해 소량만 사용)
train_dataset_gpt2 = cnn['train'].select(range(1000)).map(preprocess_gpt2_summarization, batched=True, remove_columns=cnn['train'].column_names)
eval_dataset_gpt2 = cnn['validation'].select(range(100)).map(preprocess_gpt2_summarization, batched=True, remove_columns=cnn['validation'].column_names)

train_dataset_gpt2.set_format(type='torch')
eval_dataset_gpt2.set_format(type='torch')

print("GPT-2 학습 데이터 준비 완료")
print(f"Train samples: {len(train_dataset_gpt2)}")
print(f"Eval samples: {len(eval_dataset_gpt2)}")

In [ ]:
# 3-4) GPT-2 파인튜닝 실행
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer_gpt2,
    mlm=False  # GPT-2는 causal LM
)

training_args_gpt2 = TrainingArguments(
    output_dir='./results_gpt2_cnn',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir='./logs_gpt2',
    logging_steps=50,
    load_best_model_at_end=True,
)

trainer_gpt2 = Trainer(
    model=gpt2_base,
    args=training_args_gpt2,
    train_dataset=train_dataset_gpt2,
    eval_dataset=eval_dataset_gpt2,
    data_collator=data_collator,
)

print("=== GPT-2 파인튜닝 시작 ===")
trainer_gpt2.train()

In [ ]:
# 3-5) 파인튜닝 후 GPT-2 요약 생성 및 비교
from transformers import AutoModelForCausalLM

print("\n=== 학습된 GPT-2 모델 가중치 로드 ===")
# 학습 중 저장된 best model 로드
best_gpt2_model_path = './results_gpt2_cnn/checkpoint-750'  # 실제 체크포인트 경로로 변경 필요
gpt2_finetuned = AutoModelForCausalLM.from_pretrained(best_gpt2_model_path)
gpt2_finetuned.to(device)
print(f"GPT-2 모델 로드 완료: {best_gpt2_model_path}")

print("\n=== 파인튜닝 후 GPT-2 요약 결과 ===\n")
print("[원본 Article (처음 300자)]")
print(test_article[:300])
print("\n...(생략)...\n")

gen_out_after = gpt2_finetuned.generate(
    **inputs_gpt2,
    max_length=inputs_gpt2['input_ids'].shape[1] + 100,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    return_dict_in_generate=True,
    output_attentions=True,
    pad_token_id=tokenizer_gpt2.eos_token_id
)
generated_after = tokenizer_gpt2.decode(gen_out_after.sequences[0][inputs_gpt2['input_ids'].shape[1]:], skip_special_tokens=True)

print("[정답 Highlights]")
print(cnn['test'][0]['highlights'])

print("\n[파인튜닝 전 GPT-2 생성 요약]")
print(generated_before)

print("\n[파인튜닝 후 GPT-2 생성 요약]")
print(generated_after)

In [ ]:
# 3-6) Self-Attention 시각화 (파인튜닝 후, Input vs 생성된 토큰)
# SDPA(Scaled Dot Product Attention) 때문에 attention이 None으로 반환됨
# 해결: attn_implementation을 'eager'로 변경

print("[정보] GPT-2의 Input(프롬프트) vs 생성된 토큰을 비교 시각화합니다.\n")

full_tokens = tokenizer_gpt2.convert_ids_to_tokens(gen_out_after.sequences[0].cpu().tolist())
prompt_length = inputs_gpt2['input_ids'].shape[1]  # 프롬프트 길이

print(f"전체 시퀀스 길이: {len(full_tokens)}")
print(f"프롬프트 길이: {prompt_length}")
print(f"생성된 토큰 수: {len(full_tokens) - prompt_length}")

# Input 토큰: 처음부터 프롬프트 끝까지
input_tokens = full_tokens[:prompt_length]

# 생성된 토큰: 프롬프트 이후
generated_tokens = full_tokens[prompt_length:]

# 시각화할 토큰 수 제한 (너무 많으면 보기 어려움)
max_input_tokens = 50  # Input에서 마지막 50개만
max_gen_tokens = 30    # 생성된 것 중 처음 30개만

input_start_idx = max(0, prompt_length - max_input_tokens)
gen_end_idx = min(len(generated_tokens), max_gen_tokens)

print(f"시각화할 Input 토큰: {prompt_length - input_start_idx}개 (position {input_start_idx}~{prompt_length-1})")
print(f"시각화할 생성 토큰: {gen_end_idx}개\n")

# eager 모드로 임시 모델 생성
print("임시 모델 로드 중 (eager attention)...")
import torch
from transformers import GPT2LMHeadModel

gpt2_temp = GPT2LMHeadModel.from_pretrained(
    'gpt2',
    attn_implementation="eager"
).to(device)

# 파인튜닝된 가중치 복사
gpt2_temp.load_state_dict(gpt2_finetuned.state_dict())
gpt2_temp.eval()

# 전체 시퀀스에 대한 attention 계산
with torch.no_grad():
    full_outputs = gpt2_temp(
        input_ids=gen_out_after.sequences,
        attention_mask=torch.ones_like(gen_out_after.sequences),
        output_attentions=True,
        return_dict=True
    )

# Attention 확인
if full_outputs.attentions and full_outputs.attentions[0] is not None:
    # 마지막 레이어의 전체 헤드 평균
    last_layer_attn = full_outputs.attentions[-1].detach().cpu().numpy()  # (batch, num_heads, seq_len, seq_len)

    # batch 차원 제거
    if last_layer_attn.ndim == 4:
        last_layer_attn = last_layer_attn[0]  # (num_heads, seq_len, seq_len)

    attn_mean = last_layer_attn.mean(axis=0)  # (seq_len, seq_len)

    # Y축: 생성된 토큰들 (프롬프트 이후)
    # X축: Input 토큰들 (프롬프트의 마지막 부분)
    gen_slice_end = min(prompt_length + gen_end_idx, len(full_tokens))

    # 생성된 토큰들이 Input을 어떻게 참조하는지 추출
    attn_slice = attn_mean[prompt_length:gen_slice_end, input_start_idx:prompt_length]

    tokens_y = full_tokens[prompt_length:gen_slice_end]  # 생성된 토큰 (Y축)
    tokens_x = full_tokens[input_start_idx:prompt_length]  # Input 토큰 (X축)

    print(f"=== Self-Attention 시각화 정보 ===")
    print(f"레이어 수: {len(full_outputs.attentions)}")
    print(f"헤드 수: {last_layer_attn.shape[0]}")
    print(f"Y축 (생성된 토큰): {len(tokens_y)}개")
    print(f"X축 (Input 토큰): {len(tokens_x)}개")
    print(f"Attention 행렬 shape: {attn_slice.shape}\n")

    plot_attention_heatmap(attn_slice, tokens_x, tokens_y,
                           title=f'GPT-2 Self-Attention: Generated Tokens → Input (avg of {last_layer_attn.shape[0]} heads)')

    # 임시 모델 정리
    del gpt2_temp
    torch.cuda.empty_cache()
else:
    print("[에러] 여전히 Attention을 추출할 수 없습니다.")